# 05 — Interim summary of all four JEV screening arms

**Purpose:** give colleagues one account of binary and hybrid NOUL results for JEV-9B and JEV-27B v1.1, the work already performed, and decisions needed before validation on **2,000 unseen studies**. This is an exploratory development-cohort report, not final validation.

This notebook reads the four archive sources named in the comparison YAML. It makes **no API calls**, needs no API key, and does not modify inference, input or recovery records. Saved responses are reused; missing scores stay missing. Restart the project `.venv` kernel after code changes and run top to bottom.

## 1. Load the four-arm interim report

The report configuration names `configs/evaluations/comparison.yaml`; this explicitly selects both approaches independently of `default_approach`. It verifies the common cohort and reference data, and preserves each arm's original identity and provenance. Outputs are in memory unless the final export is enabled.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from jev_eval.environment import find_project_root
from jev_eval.summary import load_interim_summary

SUMMARY_CONFIG = "configs/evaluations/interim_summary.yaml"
summary = load_interim_summary(find_project_root(), SUMMARY_CONFIG)
comparison = summary.comparison
if comparison is None:
    raise RuntimeError("Set comparison_config in the summary YAML to the four-arm comparison file.")
print("Report generated (UTC):", summary.provenance["generated_at"])
print("INTERIM / EXPLORATORY: colleague review pending; no unseen-study validation")
display(summary.tables["review_status"])
display(summary.tables["arms"][["arm", "run_id", "status", "prompt_version", "source_analysis"]])

## 2. Workflow and study flow across all four arms

The workflow separates environment checks (00), preparation (01), synthetic connectivity (02), inference (03), statistical evaluation (04) and this interim report (05). Both model deployments have binary and hybrid arms. Existing full-run artifacts are reused rather than resubmitted.

The same prepared cohort and human-reference data are required for every arm. Report per-arm scored counts and missingness; do not infer completeness from a finished-run status. Unscored records remain in the manual-review queue.

In [ ]:
display(summary.tables["study_flow"])
display(summary.tables["reference_summary"].T.rename(columns={0: "value"}))
display(
    summary.tables["inference_coverage"][
        [
            "arm",
            "successful",
            "failed",
            "unknown",
            "not_inferred_no_text",
            "missing",
            "status",
        ]
    ]
)

## 3. Preparation and human-reference audit

The source workbook is read without modification. Study ID, title and abstract form model states; human Inclusion Status, ITEM_ID and context annotations stay separate. Original study ID links predictions to labels. Missing abstracts can leave usable title-only inputs; completely textless records stay unscored.

Workbook discrepancy findings are reported without automatically changing decisions. Human-reference/adjudication procedures and unresolved label differences require author review before publication.

In [ ]:
display(summary.tables["preparation_checks"])
display(summary.tables["audit_summary"].T.rename(columns={0: "value"}))
display(summary.tables["audit_issues"])

## 4. Models, question designs and actual comparability

**Model deployment** and **question design** are separate dimensions. JEV-9B (`autotrust/jev-9b`) and JEV-27B v1.1 (`autotrust/jev-27b-v1-1`) answer structured decision questions. A NOUL is a natural-language yes/no question returning P(true). Its output is a score for the question, not proof of eligibility or a verified model-weight revision.

Binary uses one integrated overall screening question. Hybrid asks seven questions: primary research, animal study, in vivo experiment, depression model, control group, eligible outcomes and overall eligibility. Hybrid is a question design, not a third model. Primary tables use its overall answer; secondary R2/R3 analyses combine its first four domain scores.

**The saved binary and hybrid runs have different overall eligibility wording.** Preserve that provenance: these four archives provide a descriptive comparison, not a controlled estimate of question design alone. The table checks the definitions rather than assuming they match. Retain means full-text assessment, including uncertain cases, not final inclusion in the review.

In [ ]:
display(summary.tables["comparability"])
print("Archived task fingerprints:")
display(summary.tables["arms"][["arm", "task_sha256", "overall_definition_sha256"]])

### 4.1 Exact archived instructions

These are the questions actually saved with each arm. Current templates are not substituted for older binary instructions. Shared question IDs do not establish identical wording. The tables include all hybrid criteria so colleagues can review the operational definitions.

In [ ]:
for row in summary.tables["archived_criteria"].itertuples(index=False):
    display(Markdown(f"**{row.arm} / {row.question_id}**"))
    print(row.instructions)
    print("Retain:", row.retain)
    print("Exclude:", row.exclude)

## 5. Hosted inference and recorded exceptions

Metadata below describe the original saved experiments, not fresh execution during this report. Responses are retained with task IDs, available token usage and timestamps. Unknown provider outcomes deliberately skipped after review remain unscored local failures, not model exclusion decisions. Missing prediction rows remain visible even if a successful raw receipt is available.

Requested and returned model names are retained separately. App names and versions do not verify immutable model weights; unreported revisions stay null. A successful saved response should not be requested again merely to recreate a report.

In [ ]:
display(
    summary.tables["model_execution"][
        [
            "approach",
            "run_id",
            "requested_app",
            "returned_models",
            "status",
            "started_at",
            "ended_at",
            "recorded_attempts",
            "reported_input_tokens",
            "provider_reported_revisions",
            "immutable_revision_verified",
        ]
    ]
)

## 6. Screening performance across the four arms

Each arm has its own exploratory highest cutoff achieving the full-labelled recall target, with ties included, plus a 0.5 baseline. A 0.5 cutoff need not achieve high recall. No final cutoff is adopted by this report. Cutoffs selected and assessed on the same cohort are exploratory, not independent validation.

Precision/PPV, specificity and F2 use scored labelled pairs. Full-cohort recall gives unscored studies no model credit; a separate manual-retention scenario is available in the underlying tables. Cohort WSS@95 is excluded fraction minus 0.05 only when the recall requirement is met, retaining unscored studies in the review queue. It is not measured staff time, and the protocol's alternative rank expression requires reconciliation.

In [ ]:
print("Exploratory target:", comparison.settings.target_recall)
metric_columns = [
    "model",
    "cutoff_source",
    "threshold",
    "recall_scored",
    "recall_cohort_lower_bound",
    "specificity_scored",
    "ppv_scored",
    "f2_scored",
    "wss95_scored",
    "wss95_cohort_conservative",
    "review_queue_with_unscored",
    "unscored_studies",
]
performance = comparison.tables["screening_metrics"].reindex(columns=metric_columns)
display(
    performance.style.format(
        {
            "threshold": "{:.17g}",
            "recall_scored": "{:.2%}",
            "recall_cohort_lower_bound": "{:.2%}",
            "specificity_scored": "{:.2%}",
            "ppv_scored": "{:.2%}",
            "f2_scored": "{:.3f}",
            "wss95_scored": "{:.2%}",
            "wss95_cohort_conservative": "{:.2%}",
        },
        na_rep="not available",
    )
)

### 6.1 Ranking across cutoffs: ROC AUC, PR AUC and AP

AUC uses continuous scores without a decision cutoff. ROC AUC describes discrimination between human-retain and human-exclude studies. Trapezoidal PR AUC and non-interpolated average precision describe precision–recall behaviour; neither establishes probability calibration.

Available-pair results can have unequal denominators. The common-scored table restricts all four arms to the same studies; inspect its class counts too. Missing or one-class arms receive unavailable metrics, not fabricated AUCs.

In [ ]:
auc_columns = [
    "model",
    "roc_auc",
    "pr_auc_trapezoidal",
    "average_precision",
    "scored_labelled_pairs",
    "positive_pairs",
    "negative_pairs",
    "unscored_studies",
    "status",
]
display(summary.tables["ranking_metrics"][auc_columns].round(4))
print("Same all-four-scored subset:")
display(
    summary.tables["common_scored_ranking"]
    .reindex(columns=auc_columns + ["common_scored_studies"])
    .round(4)
)

### 6.2 Secondary hybrid rules and unresolved protocol decisions

R2 is the minimum and R3 the mean of the first four primary-domain scores. Controls and outcomes do not enter these rules. These are derived scores, not additional model/design arms or calibrated joint probabilities. R1 stays disabled pending clarification of its threshold-floor rule.

The pooled candidate table does not replace formal tuning using the original development/tuning membership. Reconcile the 7,008/7,000 cohort discrepancy before that step; do not invent a replacement split or silently remove studies. No winning rule or validation lock is inferred here.

In [ ]:
display(summary.tables["hybrid_derived_scores"])

## 7. Four-arm figures and paired false-negative audit

Every curve/matrix labels the NOUL approach and model. Candidate matrices use arm-specific exploratory cutoffs and scored pairs only. The paired table compares all six arm pairs among human-retain citations scored in both, reporting unavailable positives separately. It is descriptive, not a paired significance test.

In [ ]:
from jev_eval.evaluation import threshold_tradeoff_curve
from jev_eval.reporting import (
    plot_probability_curves,
    plot_recall_candidate_matrices,
    plot_threshold_tradeoffs,
)

figures = {}
roc_pr = plot_probability_curves(comparison.ranking)
if roc_pr is not None:
    figures["four-arm-roc-pr"] = roc_pr
    display(roc_pr)
else:
    print("ROC/PR curves need both human reference classes among scored pairs.")
figures["four-arm-recall-workload"] = plot_threshold_tradeoffs(
    threshold_tradeoff_curve(comparison.data),
    comparison.candidates,
    comparison.settings.target_recall,
)
display(figures["four-arm-recall-workload"])
matrices = plot_recall_candidate_matrices(comparison.candidates)
if matrices is not None:
    figures["four-arm-confusion-matrices"] = matrices
    display(matrices)
else:
    print("No attainable target cutoffs; confusion matrices withheld.")
display(summary.tables["paired_positive_errors"])

## 8. Reproducibility record and methods draft

All four original inference fingerprints, task definitions, prediction checksums and source/configuration identities are retained in report provenance. Analysis-code changes do not imply new model calls. This report's source/configuration/environment identities are recorded separately. The methods paragraph is a draft requiring author review.

In [ ]:
keys = [
    "report_type",
    "generated_at",
    "workbook_sha256",
    "conversion_manifest_sha256",
    "states_sha256",
    "reference_sha256",
    "summary_config_sha256",
    "analysis_config_sha256",
    "same_overall_eligibility_definition",
    "validation_results_present",
    "immutable_weight_revision_verified",
]
display(pd.DataFrame([{"item": key, "value": summary.provenance[key]} for key in keys]))
display(
    pd.DataFrame(summary.provenance["inference_runs"])[
        [
            "approach",
            "model",
            "run_id",
            "status",
            "original_fingerprint",
            "task_sha256",
            "predictions_sha256",
        ]
    ]
)
display(Markdown("**Methods draft — author review required**\n\n" + summary.methods))

## 9. Colleague review before unseen-study validation

Review the human-reference/adjudication procedures, archived wording differences, comparison scope, missingness, rule/threshold choice, WSS definition and original development/tuning membership. Agree the primary comparison and lock the protocol before evaluating the independent 2,000-study cohort.

No validation cohort is loaded or used here. Current performance, including pooled candidate cutoffs, cannot certify future 95% recall. Confidence intervals, paired significance and probability calibration are not established by these interim descriptive tables.

In [ ]:
display(
    Markdown(
        f"**Next:** colleague review, then locked-protocol validation on **{summary.settings.planned_validation_records:,} unseen studies**. No such results are included here."
    )
)

## 10. Optional aggregate report export — writes files

**File-writing warning:** enable both `allow_writes` in the summary YAML and `EXPORT_INTERIM_REPORT` below to write a new private HTML/CSV/PNG/PDF report. The bundle includes all four arms, coverage, metrics, wording caveats, paired audit, secondary hybrid scores and provenance. Existing input/inference files are never overwritten, and no API calls are made.

For a printout including code, use the README's notebook-05 HTML export command, then browser Print to PDF. Review outputs before sharing and keep executed notebooks and reports under ignored `results/`.

In [ ]:
from jev_eval.summary import export_interim_summary

EXPORT_INTERIM_REPORT = False
if EXPORT_INTERIM_REPORT:
    report_directory = export_interim_summary(summary, figures, allow_writes=True)
    print("Private four-arm report:", report_directory)
else:
    print("Export skipped; original inference artifacts unchanged.")